# model 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
API_key = os.getenv("GOOGLE_API_KEY")
llm_model = "gemini-2.5-flash"

In [3]:
os.environ["LANGCHAIN_PROJECT"] = "Map-reduce"

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [5]:
model = ChatGoogleGenerativeAI(
                    model=llm_model,
                    temperature=0,
                    timeout=None,
                    max_retries=2)

# Goal

- Map-reduce operations are essential for efficient task decomposition and parallel processing.

- It has two phases:

    - (1) Map - Break a task into smaller sub-tasks, processing each sub-task in parallel.

    - (2) Reduce - Aggregate the results across all of the completed, parallelized sub-tasks.

- Let's design a system that will do two things:

    - (1) Map - Create a set of jokes about a topic.

    - (2) Reduce - Pick the best joke from the list.

- We'll use an LLM to do the job generation and selection.

In [6]:
# Prompts we will use
subjects_prompt = """Generate a list of 3 sub-topics that are all related to this overall topic: {topic}."""
joke_prompt = """Generate a joke about {subject}"""
best_joke_prompt = """Below are a bunch of jokes about {topic}. Select the best one! Return the ID of the best one, starting 0 as the ID for the first joke. Jokes: \n\n  {jokes}"""


# State 

- Parallelizing joke generation
- First, let's define the entry point of the graph that will:

- Take a user input topic
- Produce a list of joke topics from it
- Send each joke topic to our above joke generation node
- Our state has a jokes key, which will accumulate jokes from parallelized joke generation

In [7]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from pydantic import BaseModel

class Subjects(BaseModel):
    subjects: list[str]

class BestJoke(BaseModel):
    id: int
    
class OverallState(TypedDict):
    topic: str
    subjects: list
    jokes: Annotated[list, operator.add]
    best_selected_joke: str

## using send 
- Send allow you to pass any state that you want to generate_joke! It does not have to align with OverallState.

In [8]:
def generate_topics(state: OverallState):
    prompt = subjects_prompt.format(topic=state["topic"])
    response = model.with_structured_output(Subjects).invoke(prompt)
    return {"subjects": response.subjects} 

In [9]:
from langgraph.types import Send
def continue_to_jokes(state: OverallState):
    return [Send("generate_joke", {"subject": s}) for s in state["subjects"]]

## Joke generation (map)

In [10]:
class JokeState(TypedDict):
    subject: str

class Joke(BaseModel):
    joke: str

def generate_joke(state: JokeState):
    prompt = joke_prompt.format(subject=state["subject"])
    response = model.with_structured_output(Joke).invoke(prompt)
    return {"jokes": [response.joke]}

## Best joke selection (reduce)

In [11]:
def best_joke(state: OverallState):
    jokes = "\n\n".join(state["jokes"])
    prompt = best_joke_prompt.format(topic=state["topic"], jokes=jokes)
    response = model.with_structured_output(BestJoke).invoke(prompt)
    return {"best_selected_joke": state["jokes"][response.id]}